# Model Context Protocol (MCP)

This notebook introduces the **Model Context Protocol (MCP)**, an open standard that lets an LLM use tools that live outside the model, such as a calculator, a web search, or your local files. You run small MCP servers that offer tools, connect a local LLM to them, and watch it decide which tool to call.

## Learning Objectives

At the end of this notebook, you should be able to:

- Connect an MCP client to a local MCP server and list the tools it offers.
- Use Ollama tool calling so that an LLM chooses and calls MCP tools.
- Compare how a small and a larger model handle the same tool-calling task.
- Apply a system prompt that makes the model prefer tool results over its training knowledge.

## Why Use MCP?

An LLM only knows what was in its training data, and by itself it cannot read your files or search today's web. **Tools** solve this, but without a standard every application wires up its tools in its own way.

MCP is a common interface. A tool is written once as an **MCP server**, and any **MCP client** can discover and call it. The alternative is to hard-code each tool inside your application. That is quicker for a single tool, but it ties the tool to one application and one model. With MCP the tools are plug and play: later in this notebook the same client function works unchanged when we swap the server.

## Setup

The notebooks run the LLM locally with **Ollama**, so no API key is needed. The next cells load the libraries and name the two models you compare:

- `ollama` sends prompts to the local model.
- `ClientSession` manages the conversation with an MCP server, and `sse_client` opens the connection to it over HTTP using Server-Sent Events (SSE).
- `SMALL_MODEL` and `LARGE_MODEL` let you run the same task twice and see how model size changes the result.

We can use any LLM with MCP, as long as it supports tool calling. In this repository we use `llama3.2:1b` and `llama3.2:3b`.

You can also try other tool-capable models from the Ollama library, if your machine supports them.

| Model | Command |
| --- | --- |
| llama 3.2:1b | `ollama pull llama3.2:1b` |
| llama 3.2:3b | `ollama pull llama3.2:3b` |

In [ ]:
from pprint import pprint

import ollama
from mcp import ClientSession
from mcp.client.sse import sse_client as http_client

In [ ]:
LARGE_MODEL = "llama3.2:3b"  # or llama3.2:1b for faster but less accurate results
SMALL_MODEL = "llama3.2:1b"  # or llama3.2:3b for better but slower results

## MCP Servers and Tools

The folder `mcp_servers` holds our MCP tools, each written as a Python script. These scripts are also called **servers**: they run and wait for a client to connect. Because MCP tools are plug and play, unlike agents, the same script can also run on a remote machine.

## Your First MCP Tool: Adding Two Numbers

We start with a very simple tool, `add_numbers`, in the script `the_math_server.py`. We use it to see how MCP works with an LLM, in this case Llama 3.2.

Start the server in a separate terminal, from the project root, and leave it running:

```bash
uv run python mcp_servers/the_math_server.py
```

In [ ]:
LOCAL_SERVER_URL = "http://127.0.0.1:54321/sse"

In [ ]:
async def run_mcp_query(user_prompt, model):
    async with http_client(LOCAL_SERVER_URL) as (read, write):
        async with ClientSession(read, write) as session:
            await session.initialize()

            # 2. List available tools from the MCP server
            tools = await session.list_tools()

            # 3. Ask Ollama to decide which tool to use
            # Note: We format the MCP tools into Ollama's tool format
            ollama_tools = [
                {
                    "type": "function",
                    "function": {
                        "name": t.name,
                        "description": t.description,
                        "parameters": t.input_schema,
                    },
                }
                for t in tools.tools
            ]

            response = ollama.chat(
                model=model,
                messages=[{"role": "user", "content": user_prompt}],
                tools=ollama_tools,
            )

            # 4. Handle Tool Calls
            if response.get("message", {}).get("tool_calls"):
                for tool_call in response["message"]["tool_calls"]:
                    tool_name = tool_call["function"]["name"]
                    tool_args = tool_call["function"]["arguments"]

                    print(f"🛠️ Calling tool: {tool_name} with {tool_args}")

                    # Execute tool via MCP
                    result = await session.call_tool(tool_name, tool_args)
                    print(f"✅ Tool Result: {result.content}")

                    # Final response from LLM
                    final_response = ollama.chat(
                        model=model,
                        messages=[
                            {"role": "user", "content": user_prompt},
                            response["message"],
                            {
                                "role": "tool",
                                "content": str(result.content),
                                "name": tool_name,
                            },
                        ],
                    )
                    return final_response["message"]["content"]

            return response["message"]["content"]

### How One Query Flows

The function `run_mcp_query` follows the same steps every time. The client asks the server which tools exist, gives those tool descriptions to the LLM together with your question, runs the tool the LLM picks, and passes the tool result back to the LLM for a final answer.

```mermaid
sequenceDiagram
    participant N as Notebook (MCP client)
    participant S as MCP server
    participant L as Ollama (LLM)
    N->>S: List available tools
    S-->>N: Tool names and schemas
    N->>L: Question plus tools
    L-->>N: Tool to call and its arguments
    N->>S: Call the tool
    S-->>N: Tool result
    N->>L: Question plus tool result
    L-->>N: Final answer
```

### Code Explanation

`async def run_mcp_query(user_prompt, MODEL):` 
- The word async (asynchronous) just means your computer won't freeze up while it waits for the process to finish. It can go do other chores, like a person putting a phone on speaker while waiting on hold.
- The Layman Translation: "I am preparing to ask my assistant a question (the user_prompt), and I'm willing to wait patiently for the answer."

`async with http_client(LOCAL_SERVER_URL) as (read, write):`
- This step builds the actual physical/digital pipe between your main program and the tool server. The `LOCAL_SERVER_URL` is like the phone number or the address of the office. `http_client` is the name this notebook gives to `sse_client`, which keeps the connection open and streams messages over HTTP.
- The Layman Translation: "I am plugging a secure phone line directly into the assistant's office. I have an earpiece to listen (read) and a microphone to speak (write)."

`async with ClientSession(read, write) as session:`
- Just having a wire isn't enough; the computers need rules on how to format their messages. A ClientSession wraps the raw wire in a polite protocol (like agreeing to say "Over" when you finish speaking on a walkie-talkie).
- The Layman Translation: "Now that the wire is connected, we are agreeing to speak the same language so we don't talk over each other."

`await session.initialize()`
- Before you can ask your actual question or ask the server to run tools, you have to do a digital handshake. This proves the server is awake, functioning, and ready to accept commands.
- The Layman Translation: "I am saying 'Hello, are you there and ready to work?' and I will wait (await) until you say 'Yes, I am ready!'"

Let us now try to use our MCP tool

In [ ]:
query = "What is 5 + 5?"

In [ ]:
result = await run_mcp_query(query, SMALL_MODEL)
print(f"LLM Final Response: {result}")

You will see above that the `SMALL_MODEL` fails to give the answer to a very simple question. Though the process successfully uses the `Math` MCP tool and the result was also calculated correctly as can be seen in the 2nd line of the output under `text='10'`.

Lets now try the same query with the `LARGE_MODEL`

In [ ]:
result = await run_mcp_query(query, LARGE_MODEL)
print(f"LLM Final Response: {result}")

### Reading the Results: Small and Large Model

These observations come from one run. Local models vary from run to run, so yours may differ.

- **Both models called the tool correctly.** Each chose `add_numbers` with `a` and `b` set to 5, and the server returned `10`. The tool itself worked.
- **They differed in what they did with the result.** The `LARGE_MODEL` answered "The answer to 5 + 5 is 10." The `SMALL_MODEL` had the right number in front of it, but replied that it could not display the result and offered to calculate it instead.

This is the key point of the notebook: MCP makes a tool available, but the model still has to read the tool result and use it well. Smaller models are weaker at that second step.

## Run a More Complex MCP Server

There is another server file, `mcp_tools_server.py`, inside the `mcp_servers` folder. It provides two tools:

1. **Web search**, using DuckDuckGo.
2. **Reading a local file**.

All servers in this repository listen on the same port, so stop the running server before you start the next one:

- Stop the current server, `the_math_server.py`, with `ctrl+c`.
- Run the `mcp_tools_server.py` with `uv run python mcp_servers/mcp_tools_server.py`

Lets first try to use the `web_search` tool

In [ ]:
# Try a web search:
query = "Who won the FIFA World Cup in 2026?"
pprint(await run_mcp_query(query, SMALL_MODEL))

In [ ]:
query = "Who won the FIFA World Cup in 2026?"
print(await run_mcp_query(query, LARGE_MODEL))

### Reading the Results: Web Search

Again, these observations come from one run.

- **The `SMALL_MODEL` passed malformed arguments.** It sent the tool's schema description (`{'type': 'object', 'required': ['query'], ...}`) instead of just the query. The server rejected the call with a validation error, and the model answered from its training data, saying it had no information about the winner.
- **The `LARGE_MODEL` called the tool correctly** with the query `FIFA World Cup 2026 winner`, and the tool returned search results. According to those results, the final was Spain against Argentina and ended 1-0 after extra time. Yet the model's answer said the final had not been played, because it trusted its training data over the tool.

That second case is a common pitfall: a model can receive correct, up-to-date tool output and still ignore it. The system prompt later in this notebook addresses it.

Now lets use the `read_local_file` tool

In [ ]:
# Try reading a file
query = "Summarize the contents of ai_response.txt"
print(await run_mcp_query(query, SMALL_MODEL))

Try to run the above code block again if facing errors or weird outputs

The `SMALL_MODEL`, that is `llama3.2:1b` does not perform very well to even not so difficult tasks

In [ ]:
# Try reading a file
query = "Summarize the contents of ai_response.txt"
pprint(await run_mcp_query(query, LARGE_MODEL))

### Reading the Results: Reading a File

- **The `SMALL_MODEL` picked the wrong tool in this run.** Instead of `read_local_file`, it called `search_web` with the query `AI`, so it never opened `ai_response.txt`.
- **The `LARGE_MODEL` called `read_local_file`** with `{'file_path': 'ai_response.txt'}` and received the contents of the file.

Choosing the right tool from several is a skill in itself, and it is where smaller models struggle most. If a cell gives an error or an odd answer, run it again.

### Add System Prompts 

A **system prompt** is a set of instructions the model follows on every request. By default, models prefer what they learned in training, even when a tool returns newer information. The alternative is to put the instructions inside the user's question, but a system prompt keeps them separate and reusable.

Now let's add a system prompt and see whether it makes a difference.

In [ ]:
async def mcp_with_system_prompt(user_prompt, model):
    async with http_client(LOCAL_SERVER_URL) as (read, write):
        async with ClientSession(read, write) as session:
            await session.initialize()

            # List tools and format for Ollama
            tools_result = await session.list_tools()
            ollama_tools = [
                {
                    "type": "function",
                    "function": {
                        "name": t.name,
                        "description": t.description,
                        "parameters": t.input_schema,
                    },
                }
                for t in tools_result.tools
            ]

            # 1. Ask Ollama
            response = ollama.chat(
                model=model,
                messages=[
                    # --- ADDED THIS SYSTEM ROLE ---
                    {
                        "role": "system",
                        "content": (
                            "You are a real-time assistant with access to tools. "
                            "IMPORTANT: Always prefer information from tool results over your internal knowledge. "
                            "The current year is 2026. If tool data contradicts your training, the tool is right."
                        ),
                    },
                    {"role": "user", "content": user_prompt},
                ],
                tools=ollama_tools,
            )

            # 2. Check for tool calls
            message = response.get("message", {})
            # print(f"LLM Response: {message.get('content', '')}")
            if message.get("tool_calls"):
                # Handle all tool calls (Ollama might suggest more than one)
                tool_messages = []
                for tool_call in message["tool_calls"]:
                    name = tool_call["function"]["name"]
                    args = tool_call["function"]["arguments"]

                    print(f"🛠️ LLM decided to use: {name}")
                    result = await session.call_tool(name, args)

                    tool_messages.append(
                        {"role": "tool", "content": str(result.content), "name": name}
                    )

                # 3. Final synthesis
                final_response = ollama.chat(
                    model=model,
                    messages=[
                        {
                            "role": "system",
                            "content": (
                                "Summarize the tool results accurately only when necessary."
                                "Do not rely on your training data but use the tools to get up-to-date information."
                                "Always prefer tool results over your internal knowledge. "
                            ),
                        },
                        {"role": "user", "content": user_prompt},
                        message,
                        *tool_messages,
                    ],
                )
                return final_response["message"]["content"]

            return message["content"]

Now let's try again with both models, one by one.

In [ ]:
query = "Who won the FIFA World Cup in 2026?"
pprint(await mcp_with_system_prompt(query, SMALL_MODEL))

In [ ]:
query = "Who won the FIFA World Cup in 2026?"
pprint(await mcp_with_system_prompt(query, LARGE_MODEL))

In [ ]:
query = "Summarize the contents of the text file ./ai_response.txt"
pprint(await mcp_with_system_prompt(query, LARGE_MODEL))

### Reading the Results: System Prompt

- **`LARGE_MODEL`, web search.** With the system prompt it answered from the tool: "Spain won the 2026 FIFA World Cup, beating Argentina 1-0 after extra time in the final." Without the prompt, the same model ignored the search results and said the winner was not yet determined. The only change was the instruction to prefer tool results.
- **`SMALL_MODEL`, web search.** It still said it could not provide the winner because its training data ends in 2022, even though it had called `search_web`. A system prompt helps a capable model, but it does not rescue a weak one.
- **`LARGE_MODEL`, file.** It summarised `ai_response.txt` as a passage that explains the role of an AI Project Manager by comparing it to building a Lego castle.

If you are getting json responses try changing system prompt. 

<details>
    <summary>Try adding this (check only if needed)</summary>
    "You are an MCP Assistant. If you need information, use a tool. DO NOT write raw JSON in your response. If you call a tool, use the proper tool-call format."
</details>

### Getting simple python code from web search

In [ ]:
query = "Search for python code for calculating the first 10 numbers of the Fibonacci sequence."
print(await mcp_with_system_prompt(query, LARGE_MODEL))

### Reading the Results: Code From a Web Search

The model searched the web and returned several Python snippets for the Fibonacci sequence, such as a loop, a version with memoisation, and a generator. The exact snippets change from run to run.

Read generated code before you run it. In this run, the first snippet defined `fibonacci(n)` to return a list, but the loop under it printed `fibonacci(i)` for each `i` from 0 to 9. That prints ten lists instead of the first ten numbers once. It is a small fix, but a good reminder that answers built from tool results still need checking.

Copy paste the above code in a python file and name it as `fibonacci_numbers.py`. Create a folder `scripts` and move the python script under the `scripts` folder

Try running this Python script in the terminal to check whether it works:

```bash
uv run python scripts/fibonacci_numbers.py
```

## Summary

In this notebook you:

- Connected an MCP client to local MCP servers and listed the tools they offer.
- Used Ollama tool calling so that a Llama model chose and called MCP tools, from simple addition to web search and file reading.
- Compared a small and a larger model, and saw the small model struggle to pass valid arguments, pick the right tool, and use the result.
- Applied a system prompt that made the larger model prefer the tool result over its training knowledge.

## References & Further Reading

- [**Model Context Protocol**](https://modelcontextprotocol.io/): Official introduction and specification of MCP
- [**MCP Python SDK**](https://py.sdk.modelcontextprotocol.io/): Documentation for the library used by the servers and clients here
- [**Ollama Tool Calling**](https://docs.ollama.com/capabilities/tool-calling): How a local model chooses and calls tools
- [**Llama 3.2 on Ollama**](https://ollama.com/library/llama3.2): The model family used in this notebook
- [**Reference MCP Servers**](https://github.com/modelcontextprotocol/servers): Open-source servers for files, databases, search, and more